In [2]:
import pandas as pd
import os
import json

In [10]:
df = pd.read_csv("./dataset/dataset.csv", header=0, names=["class_label", "image_path", "post_text"])

# Diccionario para traducir etiquetas
label_translation = {
    "enchente": ["flood","heavy rain",],
    "deslizamento de terra": ["landslide"],
    "deslizamento_de_terra": ["landslide"],
    "huaico": ["huaico", "mudflow", "avalanche"],
    "seca": ["drought", "dry river"],
    "tempestade_rayo": ["electrical storm"]
}

df["labels"] = df["class_label"].map(label_translation)

df = df[df["labels"].notnull()]

# Arreglar la ruta del archivo
df["image_path"] = df["image_path"].apply(lambda x: os.path.join("datasetDisaster/images", os.path.basename(x)).replace("\\", "/"))

# Reordenar columnas
df = df[["labels", "image_path", "post_text"]]

os.makedirs("datasetDisaster", exist_ok=True)

# Guardar el nuevo CSV
df.to_csv("datasetDisaster/dataset_multilabel.csv", index=False)

In [2]:
!pip install transformers

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   --------- ------------------------------ 2.4/10.4 MB 19.2 MB/s eta 0:00:01
   ----------------------- ---------------- 6.0/10.4 MB 17.6 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 20.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 27.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from transformers import pipeline

text = [
    "Brevity is the soul of wit.",
    "Amor, ch'a nullo amato amar perdona."
]

model_ckpt = "papluca/xlm-roberta-base-language-detection"
pipe = pipeline("text-classification", model=model_ckpt)
pipe(text, top_k=1, truncation=True)


c:\Users\RISCO\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\RISCO\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RISCO\.cache\huggingface\hub\models--papluca--xlm-roberta-base-language-detection. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or 

[[{'label': 'en', 'score': 0.8889275193214417}],
 [{'label': 'it', 'score': 0.9120125770568848}]]

In [4]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

text = [
    "Brevity is the soul of wit.",
    "Amor, ch'a nullo amato amar perdona."
]

model_ckpt = "papluca/xlm-roberta-base-language-detection"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)

inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits

preds = torch.softmax(logits, dim=-1)

# Map raw predictions to languages
id2lang = model.config.id2label
vals, idxs = torch.max(preds, dim=1)
{id2lang[k.item()]: v.item() for k, v in zip(idxs, vals)}


{'en': 0.8889274001121521, 'it': 0.9120123982429504}